In [ ]:
import numpy as np
import pandas as pd
import warnings
import optuna

# Preprocessing & Evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Traditional Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Tree & Boosting Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Imbalance Handling
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# Configuration
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Libraries imported successfully!")

In [ ]:
df=pd.read_csv('data/Churn_Modelling.csv')

In [ ]:
df.info()

In [ ]:

drop_cols=['RowNumber','CustomerId','Surname']
new_df=df.drop(drop_cols,axis=1)

In [ ]:
df_encoded=pd.get_dummies(new_df,drop_first=True)

In [ ]:
# new_df['Exited'].value_counts(normalize=True)*100

In [ ]:
X=df_encoded.drop('Exited',axis=1)
y=df_encoded['Exited']

In [ ]:
X.info()
numerical_features=X.select_dtypes(include=['int64', 'float64']).columns.to_list()
print(numerical_features)

In [ ]:
scaler=StandardScaler()
ohe=OneHotEncoder(drop='first',sparse_output=False)
X[numerical_features]=scaler.fit_transform(X[numerical_features])
ohe_cols=X.select_dtypes(include=['bool']).columns.to_list()
print(ohe_cols)
X[ohe_cols]=ohe.fit_transform(X[ohe_cols])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
print(X_train.sample(5))

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_results = {}

In [ ]:
# --- Random Forest ---
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }
    model = RandomForestClassifier(**params)
    return np.mean(cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1))

print("Tuning Random Forest...")
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=40)
best_results['Random Forest'] = {'score': study_rf.best_value, 'params': study_rf.best_params}

# --- XGBoost ---
def objective_xgb(trial):
    # scale_pos_weight adjusts for 80/20 balance (~4:1 ratio)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'logloss'
    }
    model = XGBClassifier(**params)
    return np.mean(cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1))

print("Tuning XGBoost...")
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=40)
best_results['XGBoost'] = {'score': study_xgb.best_value, 'params': study_xgb.best_params}

# --- LightGBM ---
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 8, 128),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'is_unbalance': True,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    model = LGBMClassifier(**params)
    return np.mean(cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1))

print("Tuning LightGBM...")
study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=40)
best_results['LightGBM'] = {'score': study_lgbm.best_value, 'params': study_lgbm.best_params}

print("Tuning Complete!")

In [ ]:
for i in range(10,100,10):
  print(f"tuning {i}")
  study_lgbm = optuna.create_study(direction="maximize")
  study_lgbm.optimize(objective_lgbm, n_trials=i)
  print(study_lgbm.best_value)
  print(study_lgbm.best_params)

In [ ]:
print("\n" + "="*50)
print(" ALL ALGORITHMS LEADERBOARD (CV ROC-AUC)")
print("="*50)
for name, data in sorted(best_results.items(), key=lambda item: item[1]['score'], reverse=True):
    print(f"{name:<25}: CV ROC-AUC = {data['score']:.5f}")

In [ ]:
winner_name = max(best_results, key=lambda k: best_results[k]['score'])
winner_params = best_results[winner_name]['params']

print(f" Ultimate Winner: {winner_name}")
print(f"Parameters used: {winner_params}\n")

In [ ]:
if winner_name == 'Random Forest':
    final_model = RandomForestClassifier(**winner_params, class_weight='balanced', random_state=42, n_jobs=-1)
    final_model.fit(X_train, y_train)
elif winner_name == 'XGBoost':
    final_model = XGBClassifier(**winner_params, random_state=42, n_jobs=-1)
    final_model.fit(X_train, y_train)
else: # LightGBM
    final_model = LGBMClassifier(**winner_params, is_unbalance=True, random_state=42, n_jobs=-1, verbose=-1)
    final_model.fit(X_train, y_train)

# Final Test Diagnostics
test_preds = final_model.predict(X_test)
test_probs = final_model.predict_proba(X_test)[:, 1]